In [1]:
import time
import sys

sys.path.append("../")
import cv2 as cv
import numpy as np
from pathlib import Path

from scipy.spatial.transform import Rotation

import matplotlib.pyplot as plt
import plotly.graph_objects as go

from sulllam.utils.ros import ROSPublisherWrapper
from sulllam.localization.extraction.orb import ORBFeatureExtractor, ORBConfigs
from sulllam.localization.matching.bf import BFMatcherConfig, BFFeatureMatcher
from sulllam.localization.pose_estimation.eight_point_estimator import EightPointEstimatorConfig, EightPointPoseEstimator

Images:

In [2]:
image_dir = Path("/home/zhukowych/Projects/ucu/MMML/SULLLAM/data")
images = [cv.imread(image_path) for image_path in sorted(image_dir.glob("*.png"))[::5]]
len(images)

386

Camera calibration:

In [3]:
K = np.array([[533.340727445877, 0.0, 254.64689387916482],
              [0.0, 533.2556495307942, 256.4835490935692],
              [0.0, 0.0, 1.0]])

xi = np.array([[1.73241756065]])

D = np.array([-0.05972430882700243, 0.17468739202093328, 0.000737218969875311, 0.000574074894976456])

Initialize all components

In [4]:
orb_config = ORBConfigs()
orb_extrator = ORBFeatureExtractor()

bf_config = BFMatcherConfig()
bf_matcher = BFFeatureMatcher(config=bf_config)

eight_point_config = EightPointEstimatorConfig(K=np.eye(3))
eight_point_pose_estimator = EightPointPoseEstimator(config=eight_point_config)

In [5]:
ros_publisher = ROSPublisherWrapper()


In [6]:


trajectory = [
    np.array([0, 0, 0]).T
]


R_global = np.eye(3)
t_global = np.zeros(3)

previous_keypoints, previous_descriptors = orb_extrator.extract(images[0])

for i in range(1, len(images)):
    time.sleep(1)
    current_keypoints, current_descriptors = orb_extrator.extract(images[i])
    matches = bf_matcher.match(previous_descriptors, current_descriptors)

    previous_keypoints_sorted = np.array([previous_keypoints[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    current_keypoints_sorted = np.array([current_keypoints[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    undistorted_points1 = cv.omnidir.undistortPoints(previous_keypoints_sorted, K, D, xi, np.eye(3))
    undistorted_points2 = cv.omnidir.undistortPoints(current_keypoints_sorted, K, D, xi, np.eye(3))

    estimate = eight_point_pose_estimator.estimate(previous_keypoints_sorted, current_keypoints_sorted)

    R = estimate['R']
    t = estimate['t'].reshape(-1)

    t_global = R @ t_global + t
    R_global = R @ R_global

    camera_position = -R_global.T @ t_global
    trajectory.append(camera_position)

    current_orientation = Rotation.from_matrix(R_global).as_quat()
    
    # Visualizations

    current_image_pair = cv.hconcat([images[i-1], images[i]])
    images_with_matches = cv.drawMatches(images[i-1], previous_keypoints, images[i], current_keypoints, matches, None)

    ros_publisher.publish_pose(trajectory[-1], current_orientation)
    ros_publisher.publish_current_pair(current_image_pair)
    ros_publisher.publish_current_matches(images_with_matches)

    previous_keypoints, previous_descriptors = current_keypoints, current_descriptors


trajectory = np.array(trajectory)

KeyboardInterrupt: 

In [ ]:
ros_publisher.shutdown()

[ROS] Node shut down.
